## 3. Generating Synthetic Anomaly Data

You must complete the previous setup step [0-setup-cuda128.ipynb](./0-setup-cuda128.ipynb) and training step [1-training.ipynb](./1-training.ipynb).

**Important**: This is a quick example, so the generated image quality may be suboptimal. To improve quality, increase the number of training iterations in [1-training.ipynb](./1-training.ipynb).

### 3.0 Setting Up the Environment

This notebook requires the user to set the environment variable `LOCAL_PROJECT_DIR` to the path of the PAIDF AnomalyGen repo. Remember to replace `FIXME` placeholder below with the correct path.

In [ ]:
# Set `LOCAL_PROJECT_DIR` for PAIDF AnomalyGen.
LOCAL_PROJECT_DIR="FIXME"
# Set the working directory to this path for the shell.
%cd {LOCAL_PROJECT_DIR}
# Use `cd ${LOCAL_PROJECT_DIR}` if you are copy-pasting this into a terminal.

import os

os.environ["LD_LIBRARY_PATH"] = (
    f"{LOCAL_PROJECT_DIR}/anaconda3/envs/cosmos-predict2/lib/python3.12/site-packages/nvidia/cudnn/lib:"
    + os.environ.get("LD_LIBRARY_PATH", "")
)
os.environ.pop("MPLBACKEND", None)

### 3.1 Providing the Generation Configuration

We provide a `.jsonl` file that specifies all the configurations used to generate synthetic data for MeiweiPCB dataset. Each line within this file represents a single generation task.

The codebase supports various types of augmentation. One method of augmentation involves generating random combinations of argument values.

Definition on generation configurations:

| Field                               | Datatype | Default  | Description                                                                                                | Supported Values                                   |
| ----------------------------------- | -------- | -------- | ---------------------------------------------------------------------------------------------------------- | -------------------------------------------------- |
| `image_filename`                    | str      | Required | Path to clean image for inpainting.                                                                        | Valid file path                                    |
| `mask_filename`                     | str      | Required | Path to plotted mask indicating where the anomaly should grow.                                             | Valid file path                                    |
| `anomaly_type`                      | str      | None     | The anomaly type this anomaly belongs to. Format: `TEXTURE+ANOMALY_TYPE`, e.g., `"PCB+defect"`.          | Combination of texture and anomaly type            |
| `guidance`                          | float    | 1.5      | Guidance for controlling the strength of anomaly condition guidance.                                       | Any float                                          |
| `seed`                              | int      | 1        | Seed to control sampling in the initial latent noise for diffusion process.                                | Any integer                                        |
| `num_steps`                         | int      | 35       | Denoise steps executed for each data.                                                                      | Any integer                                        |
| `num_generated_images`              | int      | 1        | Number of images generated per data point. Batch operation improves efficiency while retaining randomness. | Any integer                                        |
| `crop_and_paste`                    | bool     | True     | Whether to use crop & paste flow.                                                                          | `True` / `False`                                   |
| `crop_grid_X`                       | int      | None     | Size of cropped grid in x-axis.                                                                            | Any integer or `None`                              |
| `crop_grid_Y`                       | int      | None     | Size of cropped grid in y-axis.                                                                            | Any integer or `None`                              |
| `crop_ratio`                        | float    | None     | Ratio for cropping grid relative to masked region's bounding box.                                          | Any float or `None`                                |
| `poisson_blend`                     | bool     | False    | Whether to use Poisson blending when pasting back to the clean image.                                      | `True` / `False`                                   |
| `shift_values`                      | str      | None     | Shifted value(s) for masked region. Format: comma-separated string.                                        | E.g., `"10,20"` or `None`                          |
| `rotation_angle`                    | int      | None     | Rotated angle for masked region.                                                                           | Any integer or `None`                              |
| `morph_operation`                   | str      | None     | Morphological operation applied to mask.                                                                   | `'dilate'`, `'erode'`, `'open'`, `'close'`, `None` |
| `iterative_generation_max_instance` | int      | 5        | Maximum number of instances to iteratively generate for a single image.                                    | Any integer or `None`                              |


**Note**:
- Augmentation must be used with care. For instance, some anomalies are location-dependent (e.g., bridging occurs only across IC pins). In such cases, do not use shifting unless you have confirmed that the new, shifted position is also reasonable for growing those specific anomalies.
- All fields lacking a default value must be provided in the configurations.

In [ ]:
!cat {LOCAL_PROJECT_DIR}/ag_inference/example.jsonl

### 3.2 Generation with PAIDF AnomalyGen

Ensure the following directories are available and correctly configured:
- {LOCAL_PROJECT_DIR}/data: Training dataset ([Section 0.2](./0-setup.ipynb)).
- {LOCAL_PROJECT_DIR}/checkpoints: Required pretrained modules ([Section 0.3](./0-setup.ipynb)).
- {IMAGINAIRE_OUTPUT_ROOT}/results: Directory where training results and logs will be saved.

In the command, you should set `--ag_checkpoint_dir` to: `${IMAGINAIRE_OUTPUT_ROOT}/<project>/<group>/<name>` so it matches the training configuration you defined in (Section 1.1)

**Important**
- This is a quick example, so the generated image quality may be suboptimal. To improve quality, increase the number of training iterations in [1-training.ipynb](./1-training.ipynb).
- Based on our validation on the MeiweiPCB dataset, we recommend 75,000 iterations for full training.
- For full training, the generated images in `results/MeiweiPCB/example_output` should be similar to the training images found in `datasets/MeiweiPCB/train_downsample/PCB/anomaly_image/defect`. The generated images should appear realistic, exhibiting no artifacts such as checkerboard patterns or dark shadows around the defects.
- Ensure to set `step=75000` if you are using the full training configuration.

<details>
<summary> <b> The equivalent command in the bash terminal. (Click to show) <b> </summary>

```bash
export IMAGINAIRE_OUTPUT_ROOT=./results && \
CUDA_HOME=$CONDA_PREFIX \
CUDA_VISIBLE_DEVICES=0 \
torchrun --nproc_per_node=1 -m scripts.anomaly_gen.synthetic_dataset_generation \
--config=cosmos_predict2/configs/base/ag_config.py \
--ag_checkpoint_dir=results/anomaly_gen/MeiweiPCB/MeiweiPCB_training_exp_FP32_lr0.02_bs=2_larger_guided_mask_maskconf=0.85_2B_512x512 \
--step=200 \
--input_data_path=ag_inference/example.jsonl \
--output_image_path=results/MeiweiPCB/example_output \
--seed=0 \
-- experiment=predict2_anomaly_gen_ddp_2b
```

</details>

In [ ]:
!conda run -n cosmos-predict2 \
    bash -c "CUDA_HOME=\$CONDA_PREFIX \
        CUDA_VISIBLE_DEVICES=0 \
        torchrun --nproc_per_node=1 -m scripts.anomaly_gen.synthetic_dataset_generation \
        --config=cosmos_predict2/configs/base/ag_config.py \
        --ag_checkpoint_dir=results/anomaly_gen/MeiweiPCB/MeiweiPCB_training_exp_FP32_lr0.02_bs=2_larger_guided_mask_maskconf=0.85_2B_512x512 \
        --step=200 \
        --input_data_path=ag_inference/example.jsonl \
        --output_image_path=results/MeiweiPCB/example_output \
        --seed=0 \
        -- experiment=predict2_anomaly_gen_ddp_2b"

Generation outputs are stored under `results/MeiweiPCB/example_output`.
  - Subfolders:
    - `original_image/`: Original input images.
    - `original_mask/`: Original input masks.
    - `cropped_image/`: Original crops around input masked regions.
    - `cropped_mask/`: Cropped masks aligned with each crop.
    - `annotated_image/`: Original images overlaid with cropped regions.
    - `reconstructed_image/`: Images inpainted with learned anomaly.
  - `SDG_result.csv`: Generation metadata.

**Example generation outputs** (MeiweiPCB, PCB+defect):

<font color="red">**Important**: The step 200 example is a quick demo — generated image quality may be suboptimal.
Based on our validation on the MeiweiPCB dataset, we recommend 75,000 iterations for full training. (`trainer.max_iter: 75000`)</font>

<table>
<tr>
  <th align="center">Image Name</th>
  <th align="center">Original Image</th>
  <th align="center">Original Mask</th>
  <th align="center">Cropped Image</th>
  <th align="center">Cropped Mask</th>
  <th align="center">Annotated Image</th>
  <th align="center">Reconstructed Image (step 200)</th>
  <th align="center">Reconstructed Image (step 75000)</th>
</tr>
<tr>
  <td><sub>PCB+defect_00000</sub></td>
  <td><img src="../../assets/anomaly_gen/generation_example_materials/original_image/PCB+defect_00000.png" width="120"/></td>
  <td><img src="../../assets/anomaly_gen/generation_example_materials/original_mask/PCB+defect_00000.png" width="120"/></td>
  <td><img src="../../assets/anomaly_gen/generation_example_materials/cropped_image/PCB+defect_00000_0.png" width="120"/></td>
  <td><img src="../../assets/anomaly_gen/generation_example_materials/cropped_mask/PCB+defect_00000_0.png" width="120"/></td>
  <td><img src="../../assets/anomaly_gen/generation_example_materials/annotated_image/PCB+defect_00000_0.png" width="120"/></td>
  <td align="center"><img src="../../assets/anomaly_gen/generation_example_materials/reconstructed_image/PCB+defect_00000_step200.png" width="120"/></td>
  <td align="center"><img src="../../assets/anomaly_gen/generation_example_materials/reconstructed_image/PCB+defect_00000_step75000.png" width="120"/></td>
</tr>
<tr>
  <td><sub>PCB+defect_00001</sub></td>
  <td><img src="../../assets/anomaly_gen/generation_example_materials/original_image/PCB+defect_00001.png" width="120"/></td>
  <td><img src="../../assets/anomaly_gen/generation_example_materials/original_mask/PCB+defect_00001.png" width="120"/></td>
  <td><img src="../../assets/anomaly_gen/generation_example_materials/cropped_image/PCB+defect_00001_0.png" width="120"/></td>
  <td><img src="../../assets/anomaly_gen/generation_example_materials/cropped_mask/PCB+defect_00001_0.png" width="120"/></td>
  <td><img src="../../assets/anomaly_gen/generation_example_materials/annotated_image/PCB+defect_00001_0.png" width="120"/></td>
  <td align="center"><img src="../../assets/anomaly_gen/generation_example_materials/reconstructed_image/PCB+defect_00001_step200.png" width="120"/></td>
  <td align="center"><img src="../../assets/anomaly_gen/generation_example_materials/reconstructed_image/PCB+defect_00001_step75000.png" width="120"/></td>
</tr>
<tr>
  <td><sub>PCB+defect_00002</sub></td>
  <td><img src="../../assets/anomaly_gen/generation_example_materials/original_image/PCB+defect_00002.png" width="120"/></td>
  <td><img src="../../assets/anomaly_gen/generation_example_materials/original_mask/PCB+defect_00002.png" width="120"/></td>
  <td><img src="../../assets/anomaly_gen/generation_example_materials/cropped_image/PCB+defect_00002_0.png" width="120"/></td>
  <td><img src="../../assets/anomaly_gen/generation_example_materials/cropped_mask/PCB+defect_00002_0.png" width="120"/></td>
  <td><img src="../../assets/anomaly_gen/generation_example_materials/annotated_image/PCB+defect_00002_0.png" width="120"/></td>
  <td align="center"><img src="../../assets/anomaly_gen/generation_example_materials/reconstructed_image/PCB+defect_00002_step200.png" width="120"/></td>
  <td align="center"><img src="../../assets/anomaly_gen/generation_example_materials/reconstructed_image/PCB+defect_00002_step75000.png" width="120"/></td>
</tr>
</table>

### 3.3 Evaluation on the Generated Data

We provide an evaluation script to compute the same FID used during validation.

Required arguments:
  - `--real_path`: Should match with `dataloader_train.dataset.dataset_dir` in the training config
  - `--generated_path`: Should match with `--output_image_path` in the generation command
  - `--anomaly_types`:  List of anomaly types. Must cover all `<anomaly_type>` in the generation configuration

Note: FID computation requires that both the real and generated image sets contain more than two samples per anomaly type.


In [ ]:
!conda run -n cosmos-predict2 \
    bash -c "python -m scripts.anomaly_gen.evaluate \
        --real_path datasets/MeiweiPCB/train_downsample \
        --generated_path results/MeiweiPCB/example_output \
        --anomaly_types PCB+defect"

### 3.4 Filtering on the Generated Data

We provide a filtering script to split generated samples into keep/drop buckets according to their G-IQA scores.

Required arguments:
- `--real_path`: Should match with `dataloader_train.dataset.dataset_dir` in the training config
- `--generated_path`: Should match with `--output_image_path` in the generation command
- `--anomaly_types`: List of anomaly types. Must cover all `<anomaly_type>` in the generation configuration
- `--output_path`: path to save filtering outputs
- `--drop_ratio`: Fraction of images to discard per anomaly type (e.g. `0.2` keeps 80%)

Optional arguments:  
- `--rotation_range`: Rotation angle range `min max` in degrees (e.g., `-15 15`).
- `--rotation_step`: Increment in degrees (e.g., `15` → produces -15°, 0°, 15°).  

**Note**: Use these only if you need rotation augmentation or custom resolution for KPI computation.

Outputs:

Filtering outputs are stored under <output_path> with the following structure:
```text
<output_path>/
├── keep/
│   ├── reconstructed_image/
│   ├── original_mask/
│   ├── original_image/
│   └── SDG_result.csv
├── drop/
│   ├── reconstructed_image/
│   ├── original_mask/
│   ├── original_image/
│   └── SDG_result.csv
└── filter_result.csv
```

The contents are:
- keep/: High-quality samples retained
- drop/: Discarded samples
- filter_result.csv: Summary of all generated samples with KPI scores

In [ ]:
!conda run -n cosmos-predict2 \
    bash -c "python -m scripts.anomaly_gen.filter \
        --real_path datasets/MeiweiPCB/train_downsample \
        --generated_path results/MeiweiPCB/example_output \
        --output_path results/MeiweiPCB/filter \
        --drop_ratio 0.2 \
        --anomaly_types PCB+defect"

<font color="red">**Important**: Filtering based on GIQA can help remove low-quality samples by comparing them against a reference feature distribution. However, setting the drop ratio too high may reduce anomaly diversity. GIQA should be treated as a reference metric only, not a guarantee of better downstream performance.</font>

## Next Step

You can now proceed to the next step, pseudo-labeling the generated data, which creates a structured format and provides captions: [4-pseudo-labeling.ipynb](./4-pseudo-labeling.ipynb)